**Environment**

In [1]:
# %pip install pandas numpy matplotlib seaborn scipy tabulate
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import sys
from pathlib import Path
import json

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))

# Set up automatic module reloading on every cell execution
from IPython.core.getipython import get_ipython

def auto_reload_model():
    if 'model' in sys.modules:
        importlib.reload(sys.modules['model'])

ipython = get_ipython()
if ipython:
    ipython.events.register('pre_execute', auto_reload_model)

from model import *

**Input Data**

In [2]:
# Data path
project_root = Path.cwd()
data_path = project_root / "data"

# Output location
output_location = project_root / "output"
output_location.mkdir(parents=True, exist_ok=True)

# Liabilities
liabilities_df = pd.read_csv(data_path / 'liabs.csv', parse_dates=['date'], dayfirst=True, index_col = 'date')
liabilities = Liabilities(
    dates=liabilities_df.index.to_list(),
    cashflows=liabilities_df['cashflow']
)

# German Goverment Bond (Bund) Yields
bund_yields = pd.read_csv(data_path / 'bund_yields.csv', parse_dates=['date'], index_col = 'date')
rates = Rates(bund_yields['yield'].to_numpy(), bund_yields.index.to_list())

# Spreads
spreads_df = pd.read_csv(data_path / 'credit_spreads.csv', index_col='rating')
spread_map = spreads_df['spread'].to_dict()

# Issuers
issuers_df = pd.read_csv(data_path / 'issuers.csv', dtype={'id': 'str'}, index_col='id')
issuers = Issuers(
    ids=issuers_df.index.to_list(),
    ratings=issuers_df['rating'],
    sectors=issuers_df['sector'],
    names = issuers_df['name']
)

# Bonds
bond_data = pd.read_csv(data_path / 'bonds.csv', dtype={'id': 'str', 'issuer_id': 'str'}, index_col='id')
cashflow_ladder = pd.read_csv(data_path / 'cashflow_ladder.csv', parse_dates=['date'], dayfirst=True).set_index('date')
bonds = Bonds(
    ids = bond_data.index.to_list(),
    issuer_ids = bond_data['issuer_id'],
    recoveries = bond_data['recovery'],
    cashflows = cashflow_ladder,
    issuers = issuers
)

# Transition matrix
tmatrix = pd.read_csv(data_path / 'transition_matrix.csv')
transition_matrix = TransitionMatrix(
    tmatrix = tmatrix.values,
    labels = tmatrix.columns.to_list()
)

# Economy and Sector correlations
rho_e = .24
rho_s = np.array([rho_e])

# Model object
cr_model = CreditRiskModel(
    transition_matrix = transition_matrix,
    rho_e = rho_e,
    rho_s = rho_s,
    issuer_ids = issuers.ids,
    sector_map = issuers.sectors,
    ratings_map = issuers.ratings
)
# Valuation Date
val_date = pd.to_datetime('01-01-2026', dayfirst = True)

# Bond Allocations (Notionals per Bond ID)
cdi_allocation = pd.read_csv(data_path / 'cdi_allocation.csv', index_col = 'id').iloc[:, 0]
cmbp_allocation = pd.read_csv(data_path / 'cmbp_allocation.csv', index_col = 'id').iloc[:, 0]

# Starting Cash Value
cash = 2e06
cdi_t0 = (cdi_allocation * bonds.pv(val_date, rates, spread_map)).sum()
print(cdi_t0)

# Mandate Parameters
config = FoxConfig(
    val_date = val_date.isoformat(),
    heubeck_liabilities= 297_033_196,
    r_gaap = 0.0201,
    r_ifrs = 0.039,
    mortality_buffer = 1.0559,
    cmbp_margin = 0.001,
    fee = 0.003,
    asset_buffer = 12.5e06,
    performance_cap = 50e06,
    cash = cash,
    cdi_t0 = cdi_t0
)

# CDI Object
cdi_fox = CDIMandate_Fox(
    liabilities=liabilities,
    cash=cash,
    bonds=bonds,
    cdi_allocation=cdi_allocation,
    cmbp_allocation=cmbp_allocation,
    config=config
)

247890791.78185096


**Simulate Economy**

In [3]:
# Simulate Economy
np.random.seed(123)
n_sim = 20e3
n_years = 25
sim_results = cr_model.run(n_sim, n_years)

**Base Case**

In [4]:
# Simulate CDI Solution
cdi_sim_results = cdi_fox.run(val_date, rates, spread_map, sim_results)

# save output
folder_name = "base case"
cdi_sim_results.write_output(output_location, folder_name)

**Zero Recovery**

In [5]:
# Zero Recovery Bonds
bond_zero_recovery = pd.read_csv(data_path / 'bonds_zero_recovery.csv', dtype={'id': 'str', 'issuer_id': 'str'}, index_col='id')
cashflow_ladder = pd.read_csv(data_path / 'cashflow_ladder.csv', parse_dates=['date'], dayfirst=True).set_index('date')
bonds_zero_recovery = Bonds(
    ids = bond_zero_recovery.index.to_list(),
    issuer_ids = bond_zero_recovery['issuer_id'],
    recoveries = bond_zero_recovery['recovery'],
    cashflows = cashflow_ladder,
    issuers = issuers
)

# CDI Object
cdi_fox_zero_recovery = CDIMandate_Fox(
    liabilities=liabilities,
    cash=cash,
    bonds=bonds_zero_recovery,
    cdi_allocation=cdi_allocation,
    cmbp_allocation=cmbp_allocation,
    config=config
)

# Simulate CDI Solution
cdi_sim_results_zero_recovery = cdi_fox_zero_recovery.run(val_date, rates, spread_map, sim_results)

# save output
folder_name =  "zero recovery"
cdi_sim_results_zero_recovery.write_output(output_location, folder_name)

**High Correlation**

In [6]:

# Economy and Sector correlations
rho_e_high_correlation = .6
rho_s_high_correlation = np.array([rho_e_high_correlation])

# Model object
cr_model_high_correlation = CreditRiskModel(
    transition_matrix = transition_matrix,
    rho_e = rho_e_high_correlation,
    rho_s = rho_s_high_correlation,
    issuer_ids = issuers.ids,
    sector_map = issuers.sectors,
    ratings_map = issuers.ratings
)

# Simulate Economy
np.random.seed(123)
n_sim = 20e3
n_years = 25
sim_results_high_correlation = cr_model_high_correlation.run(n_sim, n_years)

# Simulate CDI Solution
cdi_sim_results_high_correlation = cdi_fox.run(val_date, rates, spread_map, sim_results_high_correlation)

# save output
folder_name = "high correlation"
cdi_sim_results_high_correlation.write_output(output_location, folder_name)